### Criar rasters alvo para Random Forest

Este notebook cria rasters alvo com três valores:

- `-1` = NoData / fora da área válida;
- `0` = área válida que não ardeu;
- `1` = área válida que ardeu.

Estes rasters serão usados no workflow RF-LRI.

#### Imports

In [1]:
import os
from pathlib import Path

import numpy as np
import rasterio as rio

#### Caminhos

In [2]:
area = "centro"

count_dir = f"/code/data/processed/{area}/area_ardida/raster_count"
target_dir = f"/code/data/processed/{area}/area_ardida/raster_target"

ref_rst = f"/code/data/results/{area}/lr/final/res_lri.tif" # com este caminho garante que as pesudo-ausências só ficam dentro da área burnable - no data para as árear artificiais, ágia e restantes áreas n eligiveis

os.makedirs(target_dir, exist_ok=True)

#### Funções auxiliares

In [3]:
def valid_mask(arr, nodata):
    """
    Máscara de células válidas.
    Trata correctamente NoData numérico e NoData = nan.
    """

    valid = np.isfinite(arr)

    if nodata is not None:
        try:
            nodata_is_nan = np.isnan(nodata)
        except TypeError:
            nodata_is_nan = False

        if not nodata_is_nan:
            valid = valid & (arr != nodata)

    return valid


def burned_mask(arr, nodata):
    """
    Máscara de células ardidas.
    Só considera ardido o que tem contagem maior que zero.
    """

    burned = np.isfinite(arr) & (arr > 0)

    if nodata is not None:
        try:
            nodata_is_nan = np.isnan(nodata)
        except TypeError:
            nodata_is_nan = False

        if not nodata_is_nan:
            burned = burned & (arr != nodata)

    return burned

def count_to_rf_target_blocks(count_rst, ref_rst, out_rst, nodata=-1, block_size=512):
    """
    Converte um raster de contagem de área ardida num raster alvo para RF,
    processando por blocos para evitar problemas de memória.

    Valores de saída:
    -1 = NoData / fora da área válida
     0 = área válida não ardida
     1 = área válida ardida
    """

    os.makedirs(os.path.dirname(out_rst), exist_ok=True)

    if os.path.exists(out_rst):
        os.remove(out_rst)

    counts = {}

    with rio.open(ref_rst) as ref_src, rio.open(count_rst) as cnt_src:
        if ref_src.width != cnt_src.width or ref_src.height != cnt_src.height:
            raise ValueError(
                "Os rasters não têm a mesma dimensão: "
                f"ref=({ref_src.height}, {ref_src.width}) "
                f"count=({cnt_src.height}, {cnt_src.width})"
            )

        if ref_src.transform != cnt_src.transform:
            raise ValueError("Os rasters não têm a mesma geotransformação.")

        if ref_src.crs != cnt_src.crs:
            raise ValueError("Os rasters não têm o mesmo sistema de coordenadas.")

        profile = ref_src.profile.copy()
        profile.update(
            driver="GTiff",
            dtype="int16",
            nodata=nodata,
            count=1,
            compress="lzw",
            tiled=True,
            blockxsize=block_size,
            blockysize=block_size,
            BIGTIFF="YES"
        )

        ref_nodata = ref_src.nodata
        cnt_nodata = cnt_src.nodata

        print("Raster de referência:", ref_rst)
        print("NoData referência:", ref_nodata)
        print("Raster de contagem:", count_rst)
        print("NoData contagem:", cnt_nodata)

        with rio.open(out_rst, "w", **profile) as dst:
            for row_off in range(0, ref_src.height, block_size):
                win_height = min(block_size, ref_src.height - row_off)

                for col_off in range(0, ref_src.width, block_size):
                    win_width = min(block_size, ref_src.width - col_off)

                    window = rio.windows.Window(
                        col_off=col_off,
                        row_off=row_off,
                        width=win_width,
                        height=win_height
                    )

                    ref_arr = ref_src.read(1, window=window)
                    cnt_arr = cnt_src.read(1, window=window)

                    valid = valid_mask(ref_arr, ref_nodata)
                    burned = burned_mask(cnt_arr, cnt_nodata)

                    target = np.full(ref_arr.shape, nodata, dtype=np.int16)
                    target[valid] = 0
                    target[valid & burned] = 1

                    vals, cnts = np.unique(target, return_counts=True)
                    for val, cnt in zip(vals, cnts):
                        val = int(val)
                        cnt = int(cnt)
                        counts[val] = counts.get(val, 0) + cnt

                    dst.write(target, 1, window=window)

    print("Raster alvo criado:", out_rst)
    print("Valores escritos:", counts)

    return out_rst

def check_raster_values_blocks(rst):
    """
    Verifica NoData/nan e valores finitos de um raster sem o carregar todo.
    """

    n_total = 0
    n_nan = 0
    n_finite = 0
    n_nodata_numeric = 0
    sample_values = {}

    with rio.open(rst) as src:
        nodata = src.nodata

        for _, window in src.block_windows(1):
            arr = src.read(1, window=window)

            n_total += arr.size
            n_nan += int(np.isnan(arr).sum())
            n_finite += int(np.isfinite(arr).sum())

            if nodata is not None:
                try:
                    nodata_is_nan = np.isnan(nodata)
                except TypeError:
                    nodata_is_nan = False

                if not nodata_is_nan:
                    n_nodata_numeric += int((arr == nodata).sum())

            finite_vals = arr[np.isfinite(arr)]

            if finite_vals.size > 0 and len(sample_values) < 20:
                vals, cnts = np.unique(finite_vals, return_counts=True)

                for val, cnt in zip(vals, cnts):
                    if len(sample_values) >= 20:
                        break

                    val = float(val)
                    sample_values[val] = sample_values.get(val, 0) + int(cnt)

        print("\nRaster:", rst)
        print("NoData declarado:", nodata)
        print("Dimensão:", src.width, src.height)
        print("CRS:", src.crs)
        print("Total píxeis:", n_total)
        print("Píxeis nan:", n_nan)
        print("Píxeis finitos:", n_finite)
        print("Píxeis NoData numérico:", n_nodata_numeric)
        print("Amostra de valores finitos:", sample_values)

#### Criar targets

In [4]:
target_1995_2024 = Path(target_dir) / "rst_ba_1995_2024_target.tif"
target_2025 = Path(target_dir) / "rst_ba_2025_target.tif"

count_1995_2024 = Path(count_dir) / "rst_ba_1995_2024.tif"
count_2025 = Path(count_dir) / "rst_ba_2025.tif"

assert target_1995_2024 != target_2025, "ERRO: os dois targets têm o mesmo caminho!"


print("\n" + "=" * 60)
print("Criar target 1995-2024")
print("=" * 60)

count_to_rf_target_blocks(
    count_rst=str(count_1995_2024),
    ref_rst=ref_rst,
    out_rst=str(target_1995_2024),
    nodata=-1,
    block_size=512
)

print("\n" + "=" * 60)
print("Criar target 2025")
print("=" * 60)

count_to_rf_target_blocks(
    count_rst=str(count_2025),
    ref_rst=ref_rst,
    out_rst=str(target_2025),
    nodata=-1,
    block_size=512
)


Criar target 1995-2024
Raster de referência: /code/data/results/centro/lr/final/res_lri.tif
NoData referência: nan
Raster de contagem: /code/data/processed/centro/area_ardida/raster_count/rst_ba_1995_2024.tif
NoData contagem: -1.0
Raster alvo criado: /code/data/processed/centro/area_ardida/raster_target/rst_ba_1995_2024_target.tif
Valores escritos: {-1: 84386520, 0: 69233146, 1: 67129940}

Criar target 2025
Raster de referência: /code/data/results/centro/lr/final/res_lri.tif
NoData referência: nan
Raster de contagem: /code/data/processed/centro/area_ardida/raster_count/rst_ba_2025.tif
NoData contagem: -1.0
Raster alvo criado: /code/data/processed/centro/area_ardida/raster_target/rst_ba_2025_target.tif
Valores escritos: {-1: 84386520, 0: 123578129, 1: 12784957}


'/code/data/processed/centro/area_ardida/raster_target/rst_ba_2025_target.tif'

#### Verificação final

In [5]:
def check_target_blocks(rst):
    counts = {}

    with rio.open(rst) as src:
        for _, window in src.block_windows(1):
            arr = src.read(1, window=window)
            vals, cnts = np.unique(arr, return_counts=True)

            for val, cnt in zip(vals, cnts):
                val = int(val)
                cnt = int(cnt)
                counts[val] = counts.get(val, 0) + cnt

        print("Raster:", rst)
        print("NoData:", src.nodata)
        print("Dimensão:", src.width, src.height)
        print("CRS:", src.crs)
        print("Valores:", counts)

        allowed = {-1, 0, 1}
        extra = set(counts.keys()) - allowed

        if extra:
            print("ATENÇÃO: existem valores inesperados:", extra)
        else:
            print("OK: o raster só tem valores -1, 0 e 1.")

In [6]:
for rst in [target_1995_2024, target_2025]:
    print("\n" + str(rst))
    check_target_blocks(str(rst))


/code/data/processed/centro/area_ardida/raster_target/rst_ba_1995_2024_target.tif
Raster: /code/data/processed/centro/area_ardida/raster_target/rst_ba_1995_2024_target.tif
NoData: -1.0
Dimensão: 17934 12309
CRS: EPSG:3763
Valores: {-1: 84386520, 0: 69233146, 1: 67129940}
OK: o raster só tem valores -1, 0 e 1.

/code/data/processed/centro/area_ardida/raster_target/rst_ba_2025_target.tif
Raster: /code/data/processed/centro/area_ardida/raster_target/rst_ba_2025_target.tif
NoData: -1.0
Dimensão: 17934 12309
CRS: EPSG:3763
Valores: {-1: 84386520, 0: 123578129, 1: 12784957}
OK: o raster só tem valores -1, 0 e 1.
